In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt 
from stargazer.stargazer import Stargazer
import statsmodels.formula.api as smf
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
pd.set_option('display.max_columns', 30)

# Remove people with less than 1k pageviews across all languages

In [2]:
# alive, dead, all
famous_type = 'all'

In [3]:
data = pd.read_csv("../../popularity_data/paper_data/month_data_082023.csv",low_memory=False)
data.tail()

,slug,year,month,ab,ace,ady,af,als,alt,am,ami,an,ang,ann,anp,...,xmf,yi,yo,za,zea,zgh,zh,zh-classical,zh-min-nan,zh-yue,zu,birth,death,occupations,bplace_country
6309967,Ḫattušili_III,2024,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,250.0,NaN,NaN,NaN,NaN,-1267.0,-1300.0,POLITICIAN,Iraq
6309968,Ḫattušili_III,2024,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,236.0,NaN,NaN,NaN,NaN,-1267.0,-1300.0,POLITICIAN,Iraq
6309969,Ḫattušili_III,2024,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,272.0,NaN,NaN,NaN,NaN,-1267.0,-1300.0,POLITICIAN,Iraq
6309970,Ḫattušili_III,2024,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,247.0,NaN,NaN,NaN,NaN,-1267.0,-1300.0,POLITICIAN,Iraq
6309971,Ḫattušili_III,2024,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,386.0,NaN,NaN,NaN,NaN,-1267.0,-1300.0,POLITICIAN,Iraq


In [4]:
ordered_languages = list(data.columns[3:-4])
len(ordered_languages)

337

In [5]:
ordered_languages[0], ordered_languages[-1]

('ab', 'zu')

In [6]:
time_cols = ['year','month']

In [7]:
aux = data.groupby('slug')['year'].min().reset_index()
aux.columns= ['slug','first_wikipedia_year']
data = data.merge(aux, how='left')
data.head()

,slug,year,month,ab,ace,ady,af,als,alt,am,ami,an,ang,ann,anp,...,yi,yo,za,zea,zgh,zh,zh-classical,zh-min-nan,zh-yue,zu,birth,death,occupations,bplace_country,first_wikipedia_year
0,"""Weird_Al""_Yankovic",2015,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32.0,NaN,NaN,NaN,...,NaN,32.0,NaN,NaN,NaN,582.0,NaN,29.0,NaN,NaN,1959.0,NaN,SINGER,United States,2015
1,"""Weird_Al""_Yankovic",2015,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,43.0,NaN,NaN,NaN,...,NaN,34.0,NaN,NaN,NaN,531.0,NaN,35.0,NaN,NaN,1959.0,NaN,SINGER,United States,2015
2,"""Weird_Al""_Yankovic",2015,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.0,NaN,NaN,NaN,...,NaN,58.0,NaN,NaN,NaN,609.0,NaN,36.0,NaN,NaN,1959.0,NaN,SINGER,United States,2015
3,"""Weird_Al""_Yankovic",2015,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,43.0,NaN,NaN,NaN,...,NaN,48.0,NaN,NaN,NaN,783.0,NaN,27.0,NaN,NaN,1959.0,NaN,SINGER,United States,2015
4,"""Weird_Al""_Yankovic",2015,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49.0,NaN,NaN,NaN,...,NaN,51.0,NaN,NaN,NaN,686.0,NaN,40.0,NaN,NaN,1959.0,NaN,SINGER,United States,2015


In [8]:
aux['first_wikipedia_year'].value_counts()

2015    57578
2016     1062
2018      379
2017      356
2019       94
2020        2
Name: first_wikipedia_year, dtype: int64

In [9]:
slug_cols = time_cols + ['slug']

In [10]:
data["Volume"] = data.groupby(by=slug_cols)[ordered_languages].fillna(0).astype(float).sum(axis=1)

In [11]:
data = data[data["Volume"]>=1000].copy()

In [12]:
data = data[data['slug']!='Cleopatra'].copy()

In [13]:
data.loc[data['slug'].str.contains('Bernardo_OHiggins'), 'birth'] = 1778
data.loc[data['slug'].str.contains('Bernardo_OHiggins'), 'death'] = 1842
data.loc[data['slug'].str.contains('Bernardo_OHiggins'), 'occupations'] = 'POLITICIAN'

In [14]:
data.loc[data['bplace_country']=='Frankfurt','bplace_country'] = 'Germany'
data.loc[data['bplace_country']=='German empire','bplace_country'] = 'Germany'
data.loc[data['bplace_country']=='Ancient Rome','bplace_country'] = 'Italy'

data.loc[data['slug'].apply(str).str.contains("Britain"),'bplace_country'] = 'United Kingdom'
data.loc[data['slug']=='Johnny_Unitas', 'occupations'] = 'american football player'
data.loc[data['slug']=='Franklin_D._Roosevelt', 'occupations'] = 'POLITICIAN'
data.loc[(data["slug"].isin(['Bernardo_OHiggins'])),'occupations'] = 'POLITICIAN'

In [15]:
# famous_type
if famous_type == 'alive':
    data = data[(data["birth"]>=1915)]
    data = data[(data["death"].isna())]
    print(famous_type, "Birth>=1915 & death is NaN")
elif famous_type == 'dead':
    data = data[(data["birth"]<2015)]
    data = data[(data["death"]<2015)]
    print(famous_type, "Birth<2015 & Death<2015")
else:
    print(famous_type, "no changes")

all no changes


In [16]:
data['slug'].nunique()

53387

In [17]:
len(data.loc[data['slug'].str.contains('Donald_Trump')]), len(data.loc[data['slug'].str.contains('Bernardo_OHiggins')])

(206, 109)

# Calculate local 

In [18]:
slug_cols

['year', 'month', 'slug']

In [19]:
def get_language(x):
    return ordered_languages[np.argmax(x)]

data["Locality_lang"] = (data.groupby(by=slug_cols)[ordered_languages].transform(max)).apply(lambda x: get_language(x), axis=1)
data.head()

,slug,year,month,ab,ace,ady,af,als,alt,am,ami,an,ang,ann,anp,...,za,zea,zgh,zh,zh-classical,zh-min-nan,zh-yue,zu,birth,death,occupations,bplace_country,first_wikipedia_year,Volume,Locality_lang
0,"""Weird_Al""_Yankovic",2015,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32.0,NaN,NaN,NaN,...,NaN,NaN,NaN,582.0,NaN,29.0,NaN,NaN,1959.0,NaN,SINGER,United States,2015,147652.0,en
1,"""Weird_Al""_Yankovic",2015,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,43.0,NaN,NaN,NaN,...,NaN,NaN,NaN,531.0,NaN,35.0,NaN,NaN,1959.0,NaN,SINGER,United States,2015,136787.0,en
2,"""Weird_Al""_Yankovic",2015,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.0,NaN,NaN,NaN,...,NaN,NaN,NaN,609.0,NaN,36.0,NaN,NaN,1959.0,NaN,SINGER,United States,2015,105203.0,en
3,"""Weird_Al""_Yankovic",2015,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,43.0,NaN,NaN,NaN,...,NaN,NaN,NaN,783.0,NaN,27.0,NaN,NaN,1959.0,NaN,SINGER,United States,2015,107929.0,en
4,"""Weird_Al""_Yankovic",2015,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49.0,NaN,NaN,NaN,...,NaN,NaN,NaN,686.0,NaN,40.0,NaN,NaN,1959.0,NaN,SINGER,United States,2015,87426.0,en


In [20]:
local_num = (data.groupby(by=slug_cols)[ordered_languages].transform(np.max)).max(axis=1).apply(lambda x: np.log(x))

maximum_year = data.groupby(by=["year","month"])[ordered_languages].max().to_dict()


In [21]:
maximum_year['ab'][(2015,7)]

130.0

In [22]:
local_den = data.apply(lambda x: maximum_year[x['Locality_lang']][(x['year'], x['month'])], axis=1).apply(lambda x: np.log(x))

In [23]:
data["Local_raw"] = local_num/local_den
data["maximum"] = data[["year","month","Local_raw"]].groupby(by=time_cols).transform(np.max)
data["minimum"] = data[["year","month","Local_raw"]].groupby(by=time_cols).transform(np.min)
data["local"] = (data["Local_raw"]-data["minimum"])/(data["maximum"]-data["minimum"])
data.drop(columns=['maximum','minimum'], inplace=True)
data.head()

,slug,year,month,ab,ace,ady,af,als,alt,am,ami,an,ang,ann,anp,...,zgh,zh,zh-classical,zh-min-nan,zh-yue,zu,birth,death,occupations,bplace_country,first_wikipedia_year,Volume,Locality_lang,Local_raw,local
0,"""Weird_Al""_Yankovic",2015,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32.0,NaN,NaN,NaN,...,NaN,582.0,NaN,29.0,NaN,NaN,1959.0,NaN,SINGER,United States,2015,147652.0,en,0.774591,0.667401
1,"""Weird_Al""_Yankovic",2015,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,43.0,NaN,NaN,NaN,...,NaN,531.0,NaN,35.0,NaN,NaN,1959.0,NaN,SINGER,United States,2015,136787.0,en,0.767970,0.662708
2,"""Weird_Al""_Yankovic",2015,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.0,NaN,NaN,NaN,...,NaN,609.0,NaN,36.0,NaN,NaN,1959.0,NaN,SINGER,United States,2015,105203.0,en,0.766066,0.659015
3,"""Weird_Al""_Yankovic",2015,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,43.0,NaN,NaN,NaN,...,NaN,783.0,NaN,27.0,NaN,NaN,1959.0,NaN,SINGER,United States,2015,107929.0,en,0.695057,0.558646
4,"""Weird_Al""_Yankovic",2015,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49.0,NaN,NaN,NaN,...,NaN,686.0,NaN,40.0,NaN,NaN,1959.0,NaN,SINGER,United States,2015,87426.0,en,0.716594,0.586479


# Calculate global 

In [24]:
WT = data.groupby(by=time_cols)[ordered_languages].sum().T.sum().reset_index(name='total')
WT = WT.set_index(time_cols).T
WT

year           2015                                                          \
month            7             8             9             10            11   
total  1.205945e+09  1.211912e+09  1.191648e+09  1.247780e+09  1.257186e+09   

year                         2016                                            \
month            12            1             2             3             4    
total  1.203088e+09  1.530700e+09  1.380490e+09  1.388434e+09  1.335500e+09   

year                                                                         \
month            5             6             7             8             9    
total  1.316214e+09  1.300407e+09  1.306444e+09  1.294765e+09  1.249770e+09   

year   ...          2023                                            \
month  ...            8             9             10            11   
total  ...  1.651937e+09  1.535718e+09  1.625741e+09  1.602246e+09   

year                         2024                                            \
month            12            1             2             3             4    
total  1.611881e+09  1.573397e+09  1.234707e+09  1.271620e+09  1.139336e+09   

year                                                                       \
month            5             6             7            8            9    
total  1.109705e+09  1.012783e+09  1.090077e+09  237434320.0  231632798.0   

year                
month           10  
total  254218630.0  

[1 rows x 112 columns]

In [25]:
PT_df = data.groupby(by=time_cols)[ordered_languages].sum().T.reset_index()
PT_df.rename(columns={'index':'lang'}, inplace=True)
PT_df = PT_df.set_index('lang').fillna(0)
PT_df

year                2015                                                  \
month                 7           8           9           10          11   
lang                                                                       
ab                 530.0       501.0       509.0       531.0       555.0   
ace               8422.0      6972.0      5637.0      5116.0      6052.0   
ady                  0.0         0.0         0.0         0.0         0.0   
af              105361.0    129623.0    136979.0    123119.0    111918.0   
als              48340.0     43314.0     51271.0     49575.0     52481.0   
...                  ...         ...         ...         ...         ...   
zh            14278711.0  13715869.0  13581868.0  15169881.0  14962267.0   
zh-classical     14954.0     13987.0     15301.0     15431.0     17228.0   
zh-min-nan       44415.0     40495.0     37497.0     37326.0     35372.0   
zh-yue          100961.0     99829.0    112850.0     95780.0    104751.0   
zu                2868.0      3032.0      2958.0      4075.0      2918.0   

year                            2016                                      \
month                 12          1           2           3           4    
lang                                                                       
ab                 560.0       775.0       637.0       795.0       596.0   
ace               6470.0      7419.0      6362.0      6821.0      8668.0   
ady                  0.0         0.0       238.0       375.0       405.0   
af              132411.0    147014.0    155290.0    140916.0    142354.0   
als              49114.0     58280.0     53476.0     61392.0     57134.0   
...                  ...         ...         ...         ...         ...   
zh            13868562.0  17977856.0  15870095.0  18081903.0  19012497.0   
zh-classical     17406.0     17574.0     18057.0     18575.0     18429.0   
zh-min-nan       39816.0     45162.0     39944.0     46202.0     41825.0   
zh-yue           89683.0    120732.0    111958.0    121957.0    129447.0   
zu                3467.0      5037.0      8769.0      4566.0      4028.0   

year                                                                      ...  \
month                 5           6           7           8           9   ...   
lang                                                                      ...   
ab                1325.0       462.0       557.0       609.0       534.0  ...   
ace               6370.0      5873.0      5115.0      5593.0      4602.0  ...   
ady                427.0       452.0       532.0       439.0       436.0  ...   
af              145030.0    129385.0    119882.0    160860.0    151218.0  ...   
als              61102.0     54641.0     51804.0     49492.0     55069.0  ...   
...                  ...         ...         ...         ...         ...  ...   
zh            17199684.0  16802918.0  16673120.0  18238678.0  16839927.0  ...   
zh-classical     18366.0     16617.0     13381.0     12161.0     11621.0  ...   
zh-min-nan       39483.0     37800.0     38489.0     38864.0     34199.0  ...   
zh-yue          127559.0    122875.0    124511.0    115941.0    111965.0  ...   
zu                3963.0      3518.0      4086.0      4283.0      4216.0  ...   

year                2023                                                  \
month                 8           9           10          11          12   
lang                                                                       
ab               10109.0      9338.0      9495.0     10069.0     15124.0   
ace              20047.0     21132.0     21574.0     21795.0     22132.0   
ady               1531.0      1951.0      2113.0      3101.0      3454.0   
af              641925.0    536392.0    539479.0    484800.0    544561.0   
als             169091.0    147952.0    143197.0    151815.0    119558.0   
...                  ...         ...         ...         ...         ...   
zh            22055781.0  20674744.0  25191224.0  2

In [26]:
CT_df = data.set_index(['year','slug'])[ordered_languages].sum(axis=1).reset_index(name='values')
CT_df

,year,slug,values
0,2015,"""Weird_Al""_Yankovic",147652.0
1,2015,"""Weird_Al""_Yankovic",136787.0
2,2015,"""Weird_Al""_Yankovic",105203.0
3,2015,"""Weird_Al""_Yankovic",107929.0
4,2015,"""Weird_Al""_Yankovic",87426.0
...,...,...,...
4810461,2024,Ḫattušili_III,5946.0
4810462,2024,Ḫattušili_III,5190.0
4810463,2024,Ḫattušili_III,5378.0
4810464,2024,Ḫattušili_III,5157.0


In [27]:
cols = ['slug', 'year', 'month'] + ordered_languages

In [30]:
cols = ['slug', 'year'] + ordered_languages

In [ ]:
RCA = []
for each_row in data[cols].iterrows():
    slug = each_row[1]['slug']
    year = each_row[1]['year']
    if 'month' in cols:
        month = each_row[1]['month']
    
    therow = each_row[1][ordered_languages].fillna(0)
    denominator_df = (PT_df[year] / WT[year].values[0])

    if 'month' in cols:
        CTrow = CT_df[(CT_df['month']==month)].copy()
    
    CTrow = CTrow[(CTrow['year']==year) &  (CTrow['slug']==slug)]['values'].values[0]
    
    numerator = (therow/CTrow)
    compute_ = [slug,year] + list(((therow/CTrow) / (denominator_df)).fillna(0).values)
    RCA.append(compute_)

df = pd.DataFrame(RCA)
df.columns = list(np.hstack(['slug','year',ordered_languages]))
n_languages = df.set_index(['slug','year']).apply(lambda x: x>1, axis=1).sum(axis=1).reset_index(name='n_languages')
n_languages

In [ ]:
def normalize(X):
    scaler = MinMaxScaler()
    scaler.fit(X)
    return scaler.transform(X)

# % RCA
n_languages['Global_raw'] = n_languages['n_languages']/len(ordered_languages)

n_languages["max"] = n_languages[["year","Global_raw"]].groupby(by=["year"]).transform(np.max)
n_languages["min"] = n_languages[["year","Global_raw"]].groupby(by=["year"]).transform(np.min)
n_languages['global'] = (n_languages['Global_raw']-n_languages['min'])/(n_languages['max']-n_languages['min'])
n_languages.drop(columns=['max','min'], inplace=True)

In [ ]:
n_languages[(n_languages['year']==2022) & (n_languages['slug']=="Napoleon")]

In [ ]:
n_languages[(n_languages['year']==2022) & (n_languages['slug']=="Bernardo_OHiggins")]

In [ ]:
n_languages[(n_languages['year']==2022) & (n_languages['slug']=="Johnny_Unitas")]

In [ ]:
n_languages[(n_languages['year']==2022) & (n_languages['slug']=="Audie_Murphy")]

In [ ]:
n_languages['year'] = n_languages['year'].astype(int)
data['year'] = data['year'].astype(int)

In [ ]:
len(n_languages), len(data)

In [ ]:
n_languages.head()

In [ ]:
data = n_languages.merge(data, on=['slug','year'], how='left').drop_duplicates()

In [ ]:
data.head()

In [ ]:
data['slug'].nunique()

# Plot Local and Global Collective Memory

In [ ]:
data.drop(columns=ordered_languages, inplace=True)

In [ ]:
from datetime import datetime
daycollec = datetime.today().strftime('%Y%m%d')
daycollec

In [ ]:
"data/%s_clean_localglobal_%s.csv"%(daycollec, famous_type)

In [ ]:
data.to_csv("data/%s_clean_localglobal_%s.csv"%(daycollec, famous_type), index=False)
"finished..."

In [ ]:
fig, ax = plt.subplots(figsize=(3,3),dpi=350)
lang2 = 'global'
lang1 = 'local'
show = True
year = 2022
data.loc[(data["slug"].isin(['Bernardo_OHiggins'])),'occupations'] = 'POLITICIAN'

aux_2 = data[(data["year"]==year) & (data[lang1]<=1) & (data[lang2]<=1)][['year',lang2,\
                                lang1,'occupations','slug']].drop_duplicates()\
                            .groupby(['year','slug'])[[lang2,
                                lang1]].mean()
# aux_2[lang1] = np.square(-np.log(aux_2[lang1]*500))
# aux_2[lang2] = np.square(-np.log(aux_2[lang2]*500))
sns.scatterplot(data=aux_2, x=lang1, y=lang2, s=3, alpha=0.15,
                color='lightgrey',
                markers=['o','s','>','^'], 
                legend=False, ax=ax) 

ax.set_xlabel("Local")
ax.set_ylabel("Global")

# ax.axhline(aux_2[lang2].median(), linestyle='dotted', alpha=0.5, color='grey')
# ax.axvline(aux_2[lang1].median(), linestyle='dotted', alpha=0.5, color='grey')

# ,"Viola_Davis","Katie_Holmes","Ching_Siu-tung","Maria_Dulce"\
                                 # ,"David_Schwimmer","Noah_Bean", "Mía_Maestro", "Tyler_Perry"

# "Dominique_Deruddere","Joseph_Bové","Piotr_Cugowski","David_Strauss","Maria_Helena_Andrés","Bruno_Bettinelli",\
#                                  "Veronika_Dudarova","Shukurbek_Beyshenaliev","Shakira","Konstanty_Kalinowski",\
#                                  "Étienne_Gailly","Joséphine_Fodor","Shunter_Coen",'Basshunter',
# "Margaret_Thatcher","Britney_Spears","Paul_Simon","Johnny_Unitas","Elinor_Ostrom",
# 'Donald_Trump','David_Woodard','The_Weeknd','Corbin_Bleu','Catherine_Holman','Charlotte_Rampling'
including_ = ['Carmen_Costa','Ansel_Adams','Ramo_Nakajima','Aleksei_Yuryevich_German	','Elizabeth_Schuyler_Hamilton','Geoffrey_Hughes','',\
              'George_Ball','Elvis_Presley','','Henry_J._Heinz','Ada_Lovelace','Claude_Monet',\
              'Max_Planck','Carl_Linnaeus','','Alfred_Kastler','Vincent_van_Gogh','Banine'\
              ,'Napoleon','Bernardo_OHiggins','Herta_Müller','Aaron_Hernandez','Cristiano_Ronaldo','Claude_Shannon',\
              'Galla_Placidia',"Paul_Simon","Joséphine_Fodor","Shunter_Coen",'Nelson_Mandela',\
              'Misako_Tanaka',"Johnny_Unitas","Elinor_Ostrom",'Donald_Trump','David_Woodard','',\
              'Catherine_Holman','Charlotte_Rampling','']

aux_2 = data[(data["slug"].isin(including_)) &\
             (data['year']==year) ][['year',lang2,lang1,'slug','occupations']].groupby(['year','slug','occupations'])[[lang2,lang1]].mean().reset_index()
# aux_2[lang1] = np.square(-np.log(aux_2[lang1]*500))
# aux_2[lang2] = np.square(-np.log(aux_2[lang2]*500))

sns.scatterplot(data=aux_2, x=lang1, y=lang2, s=3, 
#                 hue='K',
                color='#1c2e4a',edgecolor='#1c2e4a',linewidth=0.3,
#                 style='occupation',
                markers=['o','s','>','^'], 
                legend=False, ax=ax) 

if show == True:
    for point in aux_2.iterrows():
        t = ax.text(point[1][lang1]-0.10, point[1][lang2]+0.01, str(point[1]['slug']).replace("_(footballer)","").replace("_"," ")+"\n(%s)"%point[1]['occupations'].lower(),\
                    fontsize=3, color='#1c2e4a', alpha=1, fontweight='normal')
        # t.set_bbox(dict(facecolor='white', alpha=0.5, edgecolor='white'))


ax.set_ylim(-0.02,1.02)
ax.set_xlim(-0.02,1.02)

ax.set_xticks([0, 0.5, 1.0])
ax.set_yticks([0, 0.5, 1.0])
ax.spines[['right', 'top']].set_visible(False)


ax.tick_params(axis='both', which='major', labelsize=6)
ax.tick_params(axis='both', which='minor', labelsize=6)

ax.set_xlabel("Local",fontsize=8)
ax.set_ylabel("Global",fontsize=8)

plt.tight_layout()

fig.savefig("FigurePeople_2022.svg")
# fig.savefig("Figure.svg")